In [1]:
RUN_MODE = "observed-dev"
CONTRACT_VERSION = "2.1.2"
CRAWL_RELEASE_ID = "CRAWL_20260806_03"
DATA_VERSION = "observed-dev-20260806.1"
AS_OF_DATE = "2026-08-06"
RANDOM_SEED = 42
DATA_PROVENANCE = "OBSERVED_DEVELOPMENT_ONLY"
EMPIRICAL_ANALYSIS_ALLOWED = False
PROMOTION_ALLOWED = False
DUTY_INPUT_PATH = ""
GOLD_INPUT_PATH = ""
CONTROL_SCHEMA_DIR = ""

# P4 Agent 4 · NCS Mapping Export

**Stage:** `A4-04-EXPORT` · **Mode:** `observed-dev` · **Contract:** `2.1.2`

Build six canonical Parquet and UTF-8-SIG inspection CSV pairs.

> Development-only orchestration. Empirical analysis and production promotion are disabled.

In [2]:
from pathlib import Path
import os
import sys
import pandas as pd

NCS_ROOT = Path.cwd().resolve()
if NCS_ROOT.name != 'ncs_mapping':
    raise RuntimeError('run this notebook with cwd=ncs_mapping')
sys.path.insert(0, str(NCS_ROOT / 'src'))
assert RUN_MODE == 'observed-dev'
assert DATA_PROVENANCE == 'OBSERVED_DEVELOPMENT_ONLY'
assert EMPIRICAL_ANALYSIS_ALLOWED is False and PROMOTION_ALLOWED is False
resolved_duty_input = DUTY_INPUT_PATH or os.environ.get('P4_A2_DUTY_HANDOFF', '')
resolved_gold_input = GOLD_INPUT_PATH or os.environ.get('P4_NCS_GOLD_INPUT', '')
resolved_schema_dir = CONTROL_SCHEMA_DIR or os.environ.get('P4_CONTROL_SCHEMA_DIR', '')

In [3]:
from p4_ncs.quality.stage_artifacts import sha256_file

processed_root = NCS_ROOT / 'data/processed/observed-dev/NCS_MAPPING_OBSERVED_20260806_01'
candidate_path = processed_root / 'posting_ncs_candidates.parquet'
match_path = processed_root / 'posting_ncs_matches.parquet'
input_audit = {'candidateRows': len(pd.read_parquet(candidate_path)), 'matchRows': len(pd.read_parquet(match_path)), 'candidateSha256': sha256_file(candidate_path), 'matchSha256': sha256_file(match_path)}
assert input_audit['matchRows'] == 28
input_audit

{'candidateRows': 128,
 'matchRows': 28,
 'candidateSha256': '69e7509d8986c3a783826f589df3299c67eaecbcb0166cbc6ac9cbd427259f20',
 'matchSha256': '2d21ebba955efa58c9acae98e07d5d48a04f47f94d5cd02396dbb41bf3006748'}

In [4]:
from p4_ncs.workflow.observed import run_stage

stage_manifest = run_stage('A4-04-EXPORT', root=NCS_ROOT, duty_input_path=resolved_duty_input or None, gold_input_path=resolved_gold_input or None, schema_dir=resolved_schema_dir or None)
stage_manifest

{'manifestVersion': 'stage-manifest-v1',
 'runId': 'NCS_MAPPING_OBSERVED_20260806_01',
 'runMode': 'observed-dev',
 'stageId': 'A4-04-EXPORT',
 'status': 'SUCCEEDED',
 'agentId': 'P4-A4-NCS',
 'branch': 'agent/p4-ncs-mapping-v2',
 'gitHead': 'cce6067e578cb8dc99aacaeb465439cb1ef0faa1',
 'contractVersion': '2.1.2',
 'schemaVersion': 'ncs-export-v1',
 'dataVersion': 'observed-dev-20260806.1',
 'crawlReleaseId': 'CRAWL_20260806_03',
 'dataProvenance': 'OBSERVED_DEVELOPMENT_ONLY',
 'startedAt': '2026-08-06T08:38:48.128260Z',
 'completedAt': '2026-08-06T08:38:48.128947Z',
 'empiricalAnalysisAllowed': False,
 'promotionAllowed': False,
 'inputManifestSha256': 'a305d2354a7437860dc947f4e537be76c6233cd26fe658da650d3bb07437bc30',
 'parameterSha256': 'ee2a95b4cf722fbc15def7f7c955eba2248f681ec2a54ffef33e1869c3836e36',
 'rowCounts': {'ncs_units': 13442,
  'core_ai_it_codes': 120,
  'ncs_alias_dictionary': 10,
  'posting_ncs_candidates': 128,
  'posting_ncs_matches': 28,
  'ncs_mapping_summary': 1},


In [5]:
stage_root = NCS_ROOT / 'data/runs' / RUN_MODE / 'NCS_MAPPING_OBSERVED_20260806_01' / stage_manifest['stageId']
expected_artifacts = {'stage_manifest.json', 'stage_metrics.json', 'stage_quality.csv', 'CHECKSUMS.sha256'}
actual_artifacts = {path.name for path in stage_root.iterdir() if path.is_file()}
assert actual_artifacts == expected_artifacts
termination_summary = {'stageId': stage_manifest['stageId'], 'status': stage_manifest['status'], 'rowCounts': stage_manifest['rowCounts'], 'artifacts': sorted(actual_artifacts)}
termination_summary

{'stageId': 'A4-04-EXPORT',
 'status': 'SUCCEEDED',
 'rowCounts': {'ncs_units': 13442,
  'core_ai_it_codes': 120,
  'ncs_alias_dictionary': 10,
  'posting_ncs_candidates': 128,
  'posting_ncs_matches': 28,
  'ncs_mapping_summary': 1},
 'artifacts': ['CHECKSUMS.sha256',
  'stage_manifest.json',
  'stage_metrics.json',
  'stage_quality.csv']}